In [1]:
!pip install pyspark


In [2]:
import pandas as pd

data = {
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aayush", "Ram", "Sita", "John", "Priya", "David", "Anita", None],
    "department": ["IT", "HR", "IT", "Finance", "HR", "Finance", None, "IT"],
    "salary": [60000, 45000, 70000, None, 50000, 80000, 40000, 65000],
    "age": [24, 30, 28, 35, None, 40, 26, 29]
}

df = pd.DataFrame(data)
df.to_csv("employees.csv", index=False)

print("CSV file created successfully")

CSV file created successfully


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Distributed Employee Data Analysis") \
    .getOrCreate()

spark

In [4]:
employee_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

print("Original Employee Dataset:")
employee_df.show()
employee_df.printSchema()

Original Employee Dataset:
+-----------+------+----------+-------+----+
|employee_id|  name|department| salary| age|
+-----------+------+----------+-------+----+
|          1|Aayush|        IT|60000.0|24.0|
|          2|   Ram|        HR|45000.0|30.0|
|          3|  Sita|        IT|70000.0|28.0|
|          4|  John|   Finance|   NULL|35.0|
|          5| Priya|        HR|50000.0|NULL|
|          6| David|   Finance|80000.0|40.0|
|          7| Anita|      NULL|40000.0|26.0|
|          8|  NULL|        IT|65000.0|29.0|
+-----------+------+----------+-------+----+

root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- age: double (nullable = true)



In [5]:
clean_df = employee_df.dropna(subset=["name", "department", "salary"])

clean_df = clean_df.fillna({
    "age": 0
})

print("Cleaned Dataset:")
clean_df.show()

Cleaned Dataset:
+-----------+------+----------+-------+----+
|employee_id|  name|department| salary| age|
+-----------+------+----------+-------+----+
|          1|Aayush|        IT|60000.0|24.0|
|          2|   Ram|        HR|45000.0|30.0|
|          3|  Sita|        IT|70000.0|28.0|
|          5| Priya|        HR|50000.0| 0.0|
|          6| David|   Finance|80000.0|40.0|
+-----------+------+----------+-------+----+



In [6]:
transformed_df = clean_df.select(
    "employee_id",
    "name",
    "department",
    "salary",
    "age"
).filter(clean_df.salary > 40000)

print("Transformed Dataset:")
transformed_df.show()

Transformed Dataset:
+-----------+------+----------+-------+----+
|employee_id|  name|department| salary| age|
+-----------+------+----------+-------+----+
|          1|Aayush|        IT|60000.0|24.0|
|          2|   Ram|        HR|45000.0|30.0|
|          3|  Sita|        IT|70000.0|28.0|
|          5| Priya|        HR|50000.0| 0.0|
|          6| David|   Finance|80000.0|40.0|
+-----------+------+----------+-------+----+



In [7]:
from pyspark.sql.functions import avg, count, max, min

department_salary_df = transformed_df.groupBy("department").agg(
    avg("salary").alias("average_salary"),
    count("employee_id").alias("total_employees"),
    max("salary").alias("highest_salary"),
    min("salary").alias("lowest_salary")
)

print("Department-wise Salary Analysis:")
department_salary_df.show()

Department-wise Salary Analysis:
+----------+--------------+---------------+--------------+-------------+
|department|average_salary|total_employees|highest_salary|lowest_salary|
+----------+--------------+---------------+--------------+-------------+
|        HR|       47500.0|              2|       50000.0|      45000.0|
|   Finance|       80000.0|              1|       80000.0|      80000.0|
|        IT|       65000.0|              2|       70000.0|      60000.0|
+----------+--------------+---------------+--------------+-------------+



In [8]:
print("Number of partitions:")
print(transformed_df.rdd.getNumPartitions())

Number of partitions:
1


In [9]:
department_salary_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['department], ['department, 'avg('salary) AS average_salary#87, 'count('employee_id) AS total_employees#88, 'max('salary) AS highest_salary#89, 'min('salary) AS lowest_salary#90]
+- Filter (salary#20 > cast(40000 as double))
   +- Project [employee_id#17, name#18, department#19, salary#20, age#44]
      +- Project [employee_id#17, name#18, department#19, salary#20, coalesce(nanvl(age#21, cast(null as double)), cast(0 as double)) AS age#44]
         +- Filter atleastnnonnulls(3, name#18, department#19, salary#20)
            +- Relation [employee_id#17,name#18,department#19,salary#20,age#21] csv

== Analyzed Logical Plan ==
department: string, average_salary: double, total_employees: bigint, highest_salary: double, lowest_salary: double
Aggregate [department#19], [department#19, avg(salary#20) AS average_salary#87, count(employee_id#17) AS total_employees#88L, max(salary#20) AS highest_salary#89, min(salary#20) AS lowest_salary#90]
+- Filter (salary

In [10]:
department_salary_df.write.mode("overwrite").csv("output/department_salary_analysis", header=True)

print("Output saved successfully")

Output saved successfully


In [11]:
import os

for root, dirs, files in os.walk("output"):
    for file in files:
        print(os.path.join(root, file))

output/department_salary_analysis/._SUCCESS.crc
output/department_salary_analysis/_SUCCESS
output/department_salary_analysis/part-00000-f9a19086-e114-43ea-91b5-789b10b2763d-c000.csv
output/department_salary_analysis/.part-00000-f9a19086-e114-43ea-91b5-789b10b2763d-c000.csv.crc


Final Project Explanation

This project uses Apache Spark with PySpark to analyze employee data in a distributed processing environment. The employee dataset is loaded into a Spark DataFrame, cleaned by removing missing values, transformed by filtering useful records, and aggregated to calculate salary statistics for each department.

The project demonstrates Spark concepts such as:

Data ingestion using CSV files
Data cleaning using null handling
Data transformation using filtering and selection
Data aggregation using groupBy and aggregate functions
Parallel processing using Spark partitions
DAG execution using Spark’s execution plan
Data storage by saving the processed result